# Image Processing Fundamental

**Course:** Image Processing  
**Level:** M1 CORO / DASSIP  
**Purpose:** rebuild the core concepts required before image transformation, spatial filtering, frequency-domain filtering, and segmentation.

> **Data note.** The original exercise sheet was not recovered. This notebook uses images recovered from the course data archive and reconstructs a standard fundamental laboratory around them.


## Goal

A digital image is not only something displayed on screen: it is a numerical array. This notebook develops that idea progressively.

We will learn to:

1. load and inspect images;
2. understand pixels, coordinates, shape, channels, and data types;
3. convert RGB to grayscale;
4. access pixels and regions of interest;
5. inspect RGB channels;
6. compute statistics and histograms;
7. normalize intensities;
8. add controlled noise and measure its effect.


## 0. Setup

All paths are derived from the notebook location. No machine-specific absolute path is required.


In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Reproducible random generator used later for synthetic noise.
RNG = np.random.default_rng(42)

LAB_DIR = Path.cwd().parent
IMAGE_PROCESSING_DIR = LAB_DIR.parent
DATA_DIR = IMAGE_PROCESSING_DIR / "data" / "common"
FIG_DIR = LAB_DIR / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "einstein": DATA_DIR / "einstein.png",
    "peppers": DATA_DIR / "peppers.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "grass": DATA_DIR / "grass.jpg",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
}

missing = [str(path) for path in IMAGE_FILES.values() if not path.exists()]
assert not missing, f"Missing input files: {missing}"

print("Data directory: ../../data/common")
print("Figure directory: ../outputs/figures")
print(f"Recovered images available: {len(IMAGE_FILES)}")

Data directory: ../../data/common
Figure directory: ../outputs/figures
Recovered images available: 5


## 1. What is a digital image?

A grayscale image can be represented as a 2-D function


$$I(x,y)$$

where each pixel stores one intensity value. In code, this becomes a 2-D NumPy array with shape `(height, width)`.

An RGB color image stores three values at each pixel:

$$I(x,y,c), \qquad c \in \{R,G,B\}$$

so the array shape is normally `(height, width, 3)`.


In [2]:
toy_image = np.array([
    [0, 40, 80, 120, 160],
    [20, 60, 100, 140, 180],
    [40, 80, 120, 160, 200],
    [60, 100, 140, 180, 220],
    [80, 120, 160, 220, 255],
], dtype=np.uint8)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(toy_image, cmap="gray", vmin=0, vmax=255)
ax.set_title("A 5×5 grayscale image is a matrix of intensities")
for row in range(toy_image.shape[0]):
    for col in range(toy_image.shape[1]):
        ax.text(col, row, str(toy_image[row, col]), ha="center", va="center", fontsize=8)
ax.set_xlabel("x / column")
ax.set_ylabel("y / row")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_grayscale_matrix.png", dpi=100, bbox_inches="tight")
plt.close(fig)

print("shape:", toy_image.shape)
print("dtype:", toy_image.dtype)
print("minimum / maximum:", toy_image.min(), toy_image.max())

shape: (5, 5)
dtype: uint8
minimum / maximum: 0 255


**Generated figure — 5×5 grayscale matrix visualization**

![5×5 grayscale matrix visualization](../outputs/figures/01_grayscale_matrix.png)

### Coordinate convention

In mathematics we often write a pixel as $I(x,y)$. NumPy arrays are indexed as `image[row, column]`, which corresponds to `image[y, x]`. Mixing these conventions is a common source of bugs.


## 2. Load and inspect recovered course images

Pillow loads each file, and NumPy exposes its numerical representation.


In [3]:
images = {}
for name, path in IMAGE_FILES.items():
    pil_image = Image.open(path).convert("RGB")
    images[name] = np.asarray(pil_image)

for name, image in images.items():
    print(
        f"{name:9s} | shape={str(image.shape):16s} "
        f"dtype={image.dtype} range=[{image.min()}, {image.max()}]"
    )

einstein  | shape=(256, 256, 3)    dtype=uint8 range=[0, 255]
peppers   | shape=(417, 606, 3)    dtype=uint8 range=[0, 254]
ballons   | shape=(360, 500, 3)    dtype=uint8 range=[0, 255]
grass     | shape=(240, 247, 3)    dtype=uint8 range=[0, 255]
tower     | shape=(427, 437, 3)    dtype=uint8 range=[0, 255]


In [4]:
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
for ax, (name, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")
fig.suptitle("Recovered images used in the fundamental lab", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_dataset_overview.png", dpi=100, bbox_inches="tight")
plt.close(fig)

**Generated figure — Recovered course-image overview**

![Recovered course-image overview](../outputs/figures/02_dataset_overview.png)

## 3. Dimensions, resolution, channels, and bit depth

For an RGB array `image.shape == (H, W, 3)`:

- `H` is image height in pixels;
- `W` is image width in pixels;
- `3` is the number of channels;
- `uint8` stores integers from 0 to 255, so each channel uses 8 bits.

A 24-bit RGB pixel therefore contains three 8-bit channel values.


In [5]:
peppers = images["peppers"]
height, width, channels = peppers.shape
bytes_in_array = peppers.nbytes

print(f"Width: {width} pixels")
print(f"Height: {height} pixels")
print(f"Channels: {channels}")
print(f"Pixels: {width * height:,}")
print(f"Array memory: {bytes_in_array:,} bytes ({bytes_in_array / 1024**2:.2f} MiB)")
print(f"One RGB pixel example at (y=100, x=200): {peppers[100, 200]}")

Width: 606 pixels
Height: 417 pixels
Channels: 3
Pixels: 252,702
Array memory: 758,106 bytes (0.72 MiB)
One RGB pixel example at (y=100, x=200): [164 196 232]


## 4. RGB to grayscale from first principles

A naive average treats all channels equally, but human vision is more sensitive to green than blue. A common luminance approximation is

$$Y = 0.299R + 0.587G + 0.114B.$$

The result is clipped and converted back to `uint8`.


In [6]:
def rgb_to_grayscale(rgb_image):
    """Convert an RGB uint8 image to grayscale using luminance weights."""
    rgb_float = rgb_image.astype(np.float32)
    gray = (
        0.299 * rgb_float[..., 0]
        + 0.587 * rgb_float[..., 1]
        + 0.114 * rgb_float[..., 2]
    )
    return np.clip(gray, 0, 255).astype(np.uint8)

einstein_rgb = images["einstein"]
einstein_gray = rgb_to_grayscale(einstein_rgb)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(einstein_rgb)
axes[0].set_title("RGB")
axes[1].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Grayscale luminance")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_rgb_to_grayscale.png", dpi=100, bbox_inches="tight")
plt.close(fig)

print("RGB shape:", einstein_rgb.shape)
print("Grayscale shape:", einstein_gray.shape)

RGB shape: (256, 256, 3)
Grayscale shape: (256, 256)


**Generated figure — RGB to grayscale conversion**

![RGB to grayscale conversion](../outputs/figures/03_rgb_to_grayscale.png)

## 5. Accessing and modifying pixels

Direct indexing is useful for understanding representation, but processing algorithms should normally operate on regions or complete arrays rather than Python loops over every pixel. Always edit a copy if the original image must be preserved.


In [7]:
edited = images["ballons"].copy()
y, x = 120, 250
original_pixel = edited[y, x].copy()

# Mark a small square in pure red around the selected pixel.
edited[y-5:y+6, x-5:x+6] = [255, 0, 0]

print("Original RGB value:", original_pixel)
print("Edited center RGB value:", edited[y, x])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(images["ballons"])
axes[0].scatter([x], [y], s=80, facecolors="none", edgecolors="yellow", linewidths=2)
axes[0].set_title("Original + selected pixel")
axes[1].imshow(edited)
axes[1].set_title("Edited copy")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_pixel_edit.png", dpi=100, bbox_inches="tight")
plt.close(fig)

Original RGB value: [243 218 161]
Edited center RGB value: [255   0   0]


**Generated figure — Pixel access and safe editing**

![Pixel access and safe editing](../outputs/figures/04_pixel_edit.png)

## 6. Regions of interest (ROI)

A region of interest is simply a slice of the image array. If the image is `image[y1:y2, x1:x2]`, the first interval selects rows and the second selects columns.


In [8]:
tower = images["tower"]
h, w, _ = tower.shape

# Central crop occupying half the width and half the height.
y1, y2 = h // 4, 3 * h // 4
x1, x2 = w // 4, 3 * w // 4
roi = tower[y1:y2, x1:x2]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(tower)
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2)
axes[0].add_patch(rect)
axes[0].set_title("Full image and ROI")
axes[1].imshow(roi)
axes[1].set_title(f"ROI: {roi.shape[1]}×{roi.shape[0]}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_region_of_interest.png", dpi=100, bbox_inches="tight")
plt.close(fig)

**Generated figure — Region of interest extraction**

![Region of interest extraction](../outputs/figures/05_region_of_interest.png)

## 7. RGB channels

Each RGB channel is itself a 2-D image. Displaying channels separately helps reveal which structures contribute strongly to red, green, or blue intensity.


In [9]:
red = peppers[..., 0]
green = peppers[..., 1]
blue = peppers[..., 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(peppers)
axes[0].set_title("RGB")
for ax, channel, title in zip(axes[1:], [red, green, blue], ["Red channel", "Green channel", "Blue channel"]):
    ax.imshow(channel, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_rgb_channels.png", dpi=100, bbox_inches="tight")
plt.close(fig)

print("Channel means:", {
    "R": round(float(red.mean()), 2),
    "G": round(float(green.mean()), 2),
    "B": round(float(blue.mean()), 2),
})

Channel means: {'R': 140.79, 'G': 132.66, 'B': 146.9}


**Generated figure — RGB channel decomposition**

![RGB channel decomposition](../outputs/figures/06_rgb_channels.png)

## 8. Basic image statistics

For a grayscale image, useful first summaries include minimum, maximum, mean, standard deviation, and percentiles. They do not describe spatial arrangement, but they help characterize brightness and contrast.


In [10]:
grass_gray = rgb_to_grayscale(images["grass"])

def image_statistics(gray_image):
    return {
        "min": int(gray_image.min()),
        "max": int(gray_image.max()),
        "mean": float(gray_image.mean()),
        "std": float(gray_image.std()),
        "p05": float(np.percentile(gray_image, 5)),
        "median": float(np.median(gray_image)),
        "p95": float(np.percentile(gray_image, 95)),
    }

stats = image_statistics(grass_gray)
for key, value in stats.items():
    print(f"{key:>6s}: {value:.2f}" if isinstance(value, float) else f"{key:>6s}: {value}")

   min: 0
   max: 255
  mean: 79.51
   std: 50.50
   p05: 11.00
median: 70.00
   p95: 178.00


## 9. Intensity histograms

For 8-bit grayscale data, a histogram counts how many pixels have intensities from 0 to 255.

- a narrow histogram usually indicates low contrast;
- a wide histogram indicates a broader use of the available dynamic range;
- histogram shape alone does **not** tell us where intensities occur spatially.


In [11]:
ballons_gray = rgb_to_grayscale(images["ballons"])
counts, bin_edges = np.histogram(ballons_gray.ravel(), bins=256, range=(0, 256))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[0].axis("off")
axes[1].plot(bin_edges[:-1], counts)
axes[1].set_title("256-bin intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
axes[1].set_xlim(0, 255)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_intensity_histogram.png", dpi=100, bbox_inches="tight")
plt.close(fig)

print("Histogram pixel-count check:", counts.sum(), "==", ballons_gray.size)

Histogram pixel-count check: 180000 == 180000


**Generated figure — Intensity histogram**

![Intensity histogram](../outputs/figures/07_intensity_histogram.png)

## 10. Dynamic range and normalization

A simple min-max normalization expands the observed range $[I_{min}, I_{max}]$ to $[0,255]$:

$$I_{norm} = 255\frac{I-I_{min}}{I_{max}-I_{min}}.$$

This can improve contrast when the original image uses only a limited part of the available intensity range.


In [12]:
def minmax_normalize(gray_image):
    """Stretch a grayscale image linearly to the full 8-bit range."""
    arr = gray_image.astype(np.float32)
    low, high = arr.min(), arr.max()
    if high == low:
        return np.zeros_like(gray_image)
    normalized = 255.0 * (arr - low) / (high - low)
    return np.clip(normalized, 0, 255).astype(np.uint8)

# Create a deliberately low-contrast version to make the effect visible.
low_contrast = (90 + 0.30 * ballons_gray).clip(0, 255).astype(np.uint8)
normalized = minmax_normalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 1].imshow(normalized, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("After min-max normalization")
axes[1, 0].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[1, 0].set_title("Before")
axes[1, 1].hist(normalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After")
for ax in axes[0]:
    ax.axis("off")
for ax in axes[1]:
    ax.set_xlim(0, 255)
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Pixel count")
fig.tight_layout()
fig.savefig(FIG_DIR / "08_dynamic_range_normalization.png", dpi=100, bbox_inches="tight")
plt.close(fig)

print("Before range:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After range:", int(normalized.min()), "to", int(normalized.max()))

Before range: 90 to 166
After range: 0 to 255


**Generated figure — Dynamic-range normalization**

![Dynamic-range normalization](../outputs/figures/08_dynamic_range_normalization.png)

## 11. Introducing synthetic noise

Noise is unavoidable in real acquisition systems. Here we add Gaussian noise only to understand its numerical effect; later labs will study filtering methods for reducing it.

For additive Gaussian noise:

$$g(x,y)=f(x,y)+n(x,y), \qquad n\sim\mathcal{N}(0,\sigma^2).$$


In [13]:
sigma = 20.0
noise = RNG.normal(loc=0.0, scale=sigma, size=einstein_gray.shape)
noisy_float = einstein_gray.astype(np.float32) + noise
noisy_einstein = np.clip(noisy_float, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[1].imshow(noise, cmap="gray")
axes[1].set_title(f"Gaussian noise (σ={sigma:.0f})")
axes[2].imshow(noisy_einstein, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Noisy image")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "09_gaussian_noise.png", dpi=100, bbox_inches="tight")
plt.close(fig)

**Generated figure — Controlled Gaussian noise**

![Controlled Gaussian noise](../outputs/figures/09_gaussian_noise.png)

## 12. Quantifying image differences

Two simple error measures are useful before learning more advanced image-quality metrics.

Mean absolute error:

$$MAE = \frac{1}{N}\sum_i |x_i-y_i|$$

Root mean squared error:

$$RMSE = \sqrt{\frac{1}{N}\sum_i(x_i-y_i)^2}$$

For 8-bit images, PSNR can be computed from MSE as

$$PSNR = 10\log_{10}\left(\frac{255^2}{MSE}\right).$$


In [14]:
def comparison_metrics(reference, test):
    ref = reference.astype(np.float64)
    tst = test.astype(np.float64)
    diff = ref - tst
    mae = np.mean(np.abs(diff))
    mse = np.mean(diff ** 2)
    rmse = np.sqrt(mse)
    psnr = np.inf if mse == 0 else 10 * np.log10((255.0 ** 2) / mse)
    return mae, rmse, psnr

mae, rmse, psnr = comparison_metrics(einstein_gray, noisy_einstein)
print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"PSNR: {psnr:.2f} dB")

# Reasonableness checks: noise must change the image and metrics must be finite.
assert mae > 0
assert rmse > 0
assert np.isfinite(psnr)

MAE : 15.85
RMSE: 19.85
PSNR: 22.18 dB


## 13. Saving a processed image

A reproducible notebook should save meaningful outputs explicitly rather than relying only on what is visible in memory.


In [15]:
saved_path = FIG_DIR / "einstein_noisy.png"
Image.fromarray(noisy_einstein).save(saved_path)

reloaded = np.asarray(Image.open(saved_path))
print("Saved:", f"../outputs/figures/{saved_path.name}")
print("Reloaded shape:", reloaded.shape)
print("Exact round-trip equality:", np.array_equal(noisy_einstein, reloaded))
assert np.array_equal(noisy_einstein, reloaded)

Saved: ../outputs/figures/einstein_noisy.png
Reloaded shape: (256, 256)
Exact round-trip equality: True


## 14. Checks

The following checks summarize the essential invariants developed in this lab.


In [16]:
assert peppers.ndim == 3 and peppers.shape[2] == 3
assert einstein_gray.ndim == 2
assert einstein_gray.dtype == np.uint8
assert 0 <= einstein_gray.min() <= einstein_gray.max() <= 255
assert counts.sum() == ballons_gray.size
assert normalized.min() == 0 and normalized.max() == 255

expected_figures = [
    "01_grayscale_matrix.png",
    "02_dataset_overview.png",
    "03_rgb_to_grayscale.png",
    "04_pixel_edit.png",
    "05_region_of_interest.png",
    "06_rgb_channels.png",
    "07_intensity_histogram.png",
    "08_dynamic_range_normalization.png",
    "09_gaussian_noise.png",
    "einstein_noisy.png",
]
missing_figures = [name for name in expected_figures if not (FIG_DIR / name).exists()]
assert not missing_figures, f"Missing outputs: {missing_figures}"

print("All fundamental checks passed.")
print(f"Generated {len(expected_figures)} output figures/images.")

All fundamental checks passed.
Generated 10 output figures/images.


## 15. Practical exercises

Try these without changing the reference cells above:

1. **Coordinates:** choose three pixels in `peppers.png`, print their `(R,G,B)` values, and locate them visually.
2. **Grayscale:** compare the luminance formula with a simple channel average `(R+G+B)/3`. Where are the largest differences?
3. **ROI:** extract a different ROI from `Elizabeth_Tower_London.jpg` and report its dimensions.
4. **Histogram:** compare the grayscale histograms of `grass.jpg` and `ballons.jpg`. Which uses a wider intensity range?
5. **Noise:** repeat the noise experiment with $\sigma=5$, $20$, and $50$. Track MAE, RMSE, and PSNR.
6. **Normalization:** construct a low-contrast image with another linear transform and verify that min-max normalization uses the full 0–255 range.

**Common mistakes to avoid**

- confusing `(x, y)` with NumPy `[row, column]`;
- modifying the original array when a copy was intended;
- performing arithmetic directly on `uint8` when values can overflow or underflow;
- displaying grayscale data without `cmap="gray"`;
- assuming a histogram contains spatial information.


## 16. Key takeaways

- A digital image is a numerical array; visualization is only one representation of that data.
- Grayscale images are 2-D arrays, while RGB images usually have three channels.
- `uint8` images represent each channel with values from 0 to 255.
- Pixel and ROI operations are NumPy indexing operations.
- RGB channels carry different information and can be studied independently.
- Histograms describe the distribution of intensities but not their spatial arrangement.
- Normalization changes the mapping of intensity values; it does not recover information that was never captured.
- Controlled noise experiments provide a foundation for the next lab on filtering.

**Next lab:** `Image_Transformation`.
